<a href="https://colab.research.google.com/github/qiaotracy33/LingRep/blob/main/WeeklyModules/Week10/Week10_Participation_Improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 08 Participation: Internet Sales Analysis (Improved)

Revised version of the original notebook. Main changes:

1. **Security:** the GitHub token is no longer embedded in the clone URL (where it ends up in `.git/config` and in error messages).
2. **Bug fixes:** removed the meaningless `daynumber` column (year + month + day), and fixed `.agg(..., as_index=True)`, which pandas silently ignores.
3. **Data quality:** shows *why* 2010 and 2014 are dropped (partial years), and filters without `inplace=True`.
4. **Structure:** one config cell, one cleaning cell, reusable plotting function, no `os.chdir`.
5. **Chart:** value labels, `$` in the axis, and an honest note about the trend line (only 3 data points).

## 1. Setup and configuration

In [4]:
from pathlib import Path
import base64
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

# --- Configuration: change values here, not deep in the analysis ---
OWNER, REPO = 'qiaotracy33', 'LingRep'
DATA_RELATIVE = Path('data/InternetSales.csv')
START_DATE, END_DATE = '2011-01-01', '2013-12-31'   # complete years only (see section 3)

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


## 2. Load the data

In [5]:
def clone_repo(owner: str, repo: str) -> Path:
    """Clone a private GitHub repo without putting the token in the URL."""
    target = Path(repo)
    if target.exists():
        print(f'{repo} already cloned, skipping.')
        return target
    token = userdata.get('Git_Key')
    creds = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    try:
        subprocess.run(
            ['git', '-c', f'http.extraheader=AUTHORIZATION: basic {creds}',
             'clone', f'https://github.com/{owner}/{repo}.git'],
            check=True, capture_output=True, text=True,
        )
    except subprocess.CalledProcessError as e:
        # Re-raise without the command line, which contains the encoded token.
        raise RuntimeError(f'git clone failed: {e.stderr.strip()}') from None
    return target


repo_dir = clone_repo(OWNER, REPO) if IN_COLAB else Path('.')
data_path = repo_dir / DATA_RELATIVE

internet_sales = pd.read_csv(
    data_path,
    parse_dates=['BirthDate', 'OrderDate'],
    index_col=False,
    encoding='latin1',      # file contains non-UTF-8 characters
    low_memory=False,
)

required = {'OrderDate', 'SalesAmount'}
missing = required - set(internet_sales.columns)
assert not missing, f'Missing expected columns: {missing}'

print(internet_sales.shape)
internet_sales.head()


RuntimeError: git clone failed: Cloning into 'LingRep'...
fatal: could not read Username for 'https://github.com': No such device or address

## 3. Check data coverage before filtering

The original notebook filtered out 2010 and 2014 without showing why. This table makes the reason visible: those years only contain a few days or weeks of orders, so comparing them with full years would be misleading.

In [6]:
internet_sales = internet_sales.assign(
    orderyear=internet_sales['OrderDate'].dt.year,
    orderquarter=internet_sales['OrderDate'].dt.quarter,
    ordermonth=internet_sales['OrderDate'].dt.month,
    orderday=internet_sales['OrderDate'].dt.day,
)

coverage = internet_sales.groupby('orderyear').agg(
    first_order=('OrderDate', 'min'),
    last_order=('OrderDate', 'max'),
    rows=('OrderDate', 'size'),
    total_sales=('SalesAmount', 'sum'),
)
coverage


NameError: name 'internet_sales' is not defined

## 4. Keep complete years only

In [7]:
# .between() is inclusive on both ends; .copy() avoids SettingWithCopy warnings later.
sales = internet_sales.loc[
    internet_sales['OrderDate'].between(START_DATE, END_DATE)
].copy()

print(sales['OrderDate'].min(), '->', sales['OrderDate'].max(), sales.shape)


NameError: name 'internet_sales' is not defined

## 5. Sales summaries

In [ ]:
yearly_sales = sales.groupby('orderyear')['SalesAmount'].sum()

yearly_table = yearly_sales.to_frame('SalesAmount').assign(
    YoY_growth_pct=lambda d: d['SalesAmount'].pct_change() * 100
)
yearly_table.round(1)


In [ ]:
# Selecting the column first (instead of .agg(..., as_index=True)) gives a clean result.
quarterly_sales = (
    sales.groupby(['orderyear', 'orderquarter'])['SalesAmount']
         .sum()
         .to_frame('SalesAmount')
)
quarterly_sales


## 6. Visualisation

Note on the trend line: it is fitted to only **three** yearly points, and 2012 dips below 2011, so the line is a rough summary, not a forecast. The R² in the legend and the quarterly chart below help show how much to trust it.

In [ ]:
millions = FuncFormatter(lambda x, pos: f'${x / 1e6:.0f}M')


def plot_yearly_with_trend(yearly: pd.Series):
    years = yearly.index.to_numpy()
    values = yearly.to_numpy()

    slope, intercept = np.polyfit(years, values, 1)
    fitted = slope * years + intercept
    r2 = 1 - ((values - fitted) ** 2).sum() / ((values - values.mean()) ** 2).sum()

    fig, ax = plt.subplots(figsize=(9, 5.5))
    bars = ax.bar(years, values, color='steelblue', label='Yearly sales')
    ax.bar_label(bars, labels=[f'${v / 1e6:.1f}M' for v in values], padding=3)
    ax.plot(years, fitted, '--', color='red', linewidth=2,
            label=f'Linear trend (R² = {r2:.2f}, n = {len(years)} years)')

    ax.set_xticks(years)
    ax.set_title('Internet Sales by Year with Trend Line')
    ax.set_xlabel('Year')
    ax.set_ylabel('Sales')
    ax.yaxis.set_major_formatter(millions)
    ax.legend()
    fig.tight_layout()
    return fig


plot_yearly_with_trend(yearly_sales)
plt.show()


In [ ]:
# Quarterly view: more data points, and it shows the 2013 acceleration the yearly chart hides.
q = quarterly_sales['SalesAmount']
labels = [f'{y} Q{qtr}' for y, qtr in q.index]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(labels, q.to_numpy(), marker='o', color='steelblue')
ax.set_title('Internet Sales by Quarter')
ax.set_ylabel('Sales')
ax.yaxis.set_major_formatter(millions)
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.show()


## 7. Further ideas (not applied)

- Add `SalesAmount` breakdowns by `CountryRegionCode` or product category to explain *what* drove 2013 growth.
- Save figures with `fig.savefig('yearly_sales.png', dpi=200)` for reports.
- Wrap sections 3-5 into a function if you reuse this analysis on other tables.
- Add a few `assert` checks on the cleaned data (e.g., no negative `SalesAmount`, no duplicate order lines).